In [ ]:
import dash
from dash import dcc, html
import plotly.express as px
import pandas as pd
import matplotlib.pyplot as plt

app = dash.Dash(__name__)

# ---------------- LOAD YOUR DATASET ---------------- #

df = pd.read_csv("Startup_Cleaned_Dataset.csv")
df.columns = df.columns.str.strip()

# ---------------- DATA CLEANING ---------------- #

# Fix Profit Margin (%)
df["Profit Margin (%)"] = (
    df["Profit Margin (%)"]
    .astype(str)
    .str.replace('%', '', regex=False)
)
df["Profit Margin (%)"] = pd.to_numeric(df["Profit Margin (%)"], errors='coerce')

# Fix Revenue Growth (%)
df["Revenue Growth Rate (%)"] = (
    df["Revenue Growth Rate (%)"]
    .astype(str)
    .str.replace('%', '', regex=False)
)
df["Revenue Growth Rate (%)"] = pd.to_numeric(df["Revenue Growth Rate (%)"], errors='coerce')

# ---------------- CHARTS ---------------- #

# 5) Top 10 Funded Startups
top10 = df.sort_values(by="Amount in USD", ascending=False).head(10)
fig1 = px.bar(top10, x="Startup_Name", y="Amount in USD")

# 6) Revenue vs Burn Rate
fig2 = px.scatter(df, x="Monthly Revenue", y="Annual Burn Rate")

# 7) CLV vs CAC Analysis
fig3 = px.scatter(df,
                  x="Customer Acquisition Cost (CAC)",
                  y="Customer Lifetime Value (CLV)")

# 8) Active Users vs Revenue Growth
fig4 = px.scatter(df,
                  x="Active Users",
                  y="Revenue Growth Rate (%)")

# 9) Profit Margin by Industry (Top 10 only)

top_industries = (
    df.groupby("Industry_Vertical_Std")["Profit Margin (%)"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .index
)

df_top = df[df["Industry_Vertical_Std"].isin(top_industries)]

fig5 = px.box(df_top,
              x="Industry_Vertical_Std",
              y="Profit Margin (%)")
plt.figure(figsize=(10, 7))



# ---------------- COMMON STYLING ---------------- #

for fig in [fig1, fig2, fig3, fig4, fig5]:
    fig.update_layout(
        plot_bgcolor="#F5F7FB",
        paper_bgcolor="#F5F7FB",
        font=dict(color="#2B2B2B"),
        margin=dict(l=20, r=20, t=50, b=20),
        title_font=dict(size=16),
        title={
            'x': 0.5,
            'xanchor': 'center'
        }
    )

# ---------------- LAYOUT ---------------- #

app.layout = html.Div([

    html.H2("📊 Startup Funding Dashboard",
            style={
                "textAlign": "center",
                "marginBottom": "20px",
                "color": "#1F3C88",
                "fontWeight": "600"
            }),

    html.Div([
        html.Div([
            html.H4("📈 Top Funded Startups",
                    style={
                        "textAlign": "center",
                        "width": "100%",
                        "display": "block",
                        "margin": "0 auto 10px auto",
                        "color": "#2EC4B6",
                        "fontWeight": "600"
                    }),
            dcc.Graph(figure=fig1)
        ], className="card"),

        html.Div([
            html.H4("💸 Revenue vs Burn Rate",
                    style={
                        "textAlign": "center",
                        "width": "100%",
                        "display": "block",
                        "margin": "0 auto 10px auto",
                        "color": "#2EC4B6",
                        "fontWeight": "600"
                    }),
            dcc.Graph(figure=fig2)
        ], className="card")
    ], className="row"),

    html.Div([
        html.Div([
            html.H4("📊 CLV vs CAC",
                    style={
                        "textAlign": "center",
                        "width": "100%",
                        "display": "block",
                        "margin": "0 auto 10px auto",
                        "color": "#2EC4B6",
                        "fontWeight": "600"
                    }),
            dcc.Graph(figure=fig3)
        ], className="card"),

        html.Div([
            html.H4("🚀 Users vs Growth",
                    style={
                        "textAlign": "center",
                        "width": "100%",
                        "display": "block",
                        "margin": "0 auto 10px auto",
                        "color": "#2EC4B6",
                        "fontWeight": "600"
                    }),
            dcc.Graph(figure=fig4)
        ], className="card")
    ], className="row"),

    html.Div([
        html.Div([
            html.H4("🏭 Profit Margin by Industry",
                    style={
                        "textAlign": "center",
                        "width": "100%",
                        "display": "block",
                        "margin": "0 auto 10px auto",
                        "color": "#2EC4B6",
                        "fontWeight": "600"
                    }),
            dcc.Graph(figure=fig5)
        ], className="card full-width")
    ], className="row")

])

# ---------------- STYLE ---------------- #

app.index_string = '''
<!DOCTYPE html>
<html>
<head>
<style>

body{
background:#F5F7FB;
font-family: 'Segoe UI', Arial, sans-serif;
margin:0;
padding:20px;
}

.row{
display:flex;
gap:20px;
margin-bottom:20px;
}

.card{
background:white;
padding:15px;
border-radius:12px;
box-shadow:0 4px 12px rgba(31,60,136,0.10);
flex:1;
transition: all 0.3s ease;
}

.card:hover{
transform: translateY(-5px);
box-shadow:0 8px 20px rgba(31,60,136,0.20);
}

.full-width{
flex:100%;
}

h4{
margin-bottom:10px;
color:#2EC4B6;
font-weight:600;
}

</style>
</head>

<body>
{%app_entry%}

<footer>
{%config%}
{%scripts%}
{%renderer%}
</footer>

</body>
</html>
'''

# ---------------- RUN ---------------- #

if __name__ == "__main__":
    app.run(debug=True, port=8066)